In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from fuzzywuzzy import process
import difflib

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    tickers = []
    for row in table.find_all('tr')[1:]:
        ticker = row.find_all('td')[0].text.strip().replace('.', '-')
        tickers.append(ticker)
    return sorted(tickers)

def map_tickers_to_figis(tickers, api_key):
    headers = {
        'Content-Type': 'application/json',
        'X-OPENFIGI-APIKEY': api_key
    }
    url = 'https://api.openfigi.com/v3/mapping'
    results = []

    for i in range(0, len(tickers), 100):
        batch = tickers[i:i+100]
        mapping_jobs = [{'idType': 'TICKER', 'idValue': ticker, 'exchCode': 'US'} for ticker in batch]
        response = requests.post(url, headers=headers, json=mapping_jobs)
        if response.status_code == 200:
            data = response.json()
            for i, item in enumerate(data):
                if 'data' in item and item['data']:
                    for entry in item['data']:
                        results.append({
                            'ticker': batch[i],
                            'name': entry.get('name'),
                            'figi': entry.get('figi'),
                            'cusip': entry.get('cusip')
                        })
        else:
            print(f"Error: {response.status_code} - {response.text}")
        time.sleep(1)

    override = [
        {'ticker': 'BRK-B', 'cusip': None, 'name': 'BERKSHIRE HATHAWAY INC', 'figi': 'BBG000DWG505'},
        {'ticker': 'BF-B', 'cusip': None, 'name': 'BROWN FORMAN CORP', 'figi': 'BBG001S5P4H1'}
    ]

    for item in override:
        results.append({
            'ticker': item['ticker'],
            'cusip': item['cusip'],
            'name': item['name'],
            'figi': item['tick']
        })

    return pd.DataFrame(results)

def match_names_to_issuer(openfigi_df, csv_df):
    matched = []
    csv_names = csv_df['NAMEOFISSUER'].dropna().tolist()

    for _, row in openfigi_df.iterrows():
        figi_name = row['name']
        if not figi_name:
            best_match = None
        else:
            best_match, score = process.extractOne(figi_name, csv_names)
            if score < 80:  # You can adjust the threshold
                best_match = None
        matched.append(best_match)
        print(f"Processed {figi_name}. {_}")

    openfigi_df['matched_issuer'] = matched
    return openfigi_df

df_cusip = pd.DataFrame(columns=['CUSIP', 'ticker', 'name', 'FIGI'])

# Example usage
api_key = '8c4a6a49-dc82-4baf-bdb2-27e32ab5c717'
tickers = get_sp500_tickers()
figi_df = map_tickers_to_figis(tickers, api_key)
csv_df = pd.read_csv('sp500_random.csv')

result_df = match_names_to_issuer(figi_df, csv_df)

# Check which tickers were not matched
unmatched = result_df[result_df['matched_issuer'].isna()]
print(f"Unmatched tickers: {len(unmatched)}")
print(unmatched[['ticker', 'name']])

# Optionally: Save to CSV
#.to_csv('sp500_figi_matched.csv', index=False)


C:\Users\emadi\AppData\Local\Temp\ipykernel_12128\733291971.py:83: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_df = pd.read_csv('sp500_random.csv')


Processed AGILENT TECHNOLOGIES INC. 0
Processed APPLE INC. 1
Processed ABBVIE INC. 2
Processed AIRBNB INC-CLASS A. 3
Processed ABBOTT LABORATORIES. 4
Processed ARCH CAPITAL GROUP LTD. 5
Processed ACCENTURE PLC-CL A. 6
Processed ADOBE INC. 7
Processed ANALOG DEVICES INC. 8
Processed ARCHER-DANIELS-MIDLAND CO. 9
Processed AUTOMATIC DATA PROCESSING. 10
Processed AUTODESK INC. 11
Processed AMEREN CORPORATION. 12
Processed AMERICAN ELECTRIC POWER. 13
Processed AES CORP. 14
Processed AFLAC INC. 15
Processed AMERICAN INTERNATIONAL GROUP. 16
Processed ASSURANT INC. 17
Processed ARTHUR J GALLAGHER & CO. 18
Processed AKAMAI TECHNOLOGIES INC. 19
Processed ALBEMARLE CORP. 20
Processed ALIGN TECHNOLOGY INC. 21
Processed ALLSTATE CORP. 22
Processed ALLEGION PLC. 23
Processed APPLIED MATERIALS INC. 24
Processed AMCOR PLC. 25
Processed ADVANCED MICRO DEVICES. 26
Processed AMETEK INC. 27
Processed AMGEN INC. 28
Processed AMERIPRISE FINANCIAL INC. 29
Processed AMERICAN TOWER CORP. 30
Processed AMAZON.CO

In [5]:
import pandas as pd
from fuzzywuzzy import process

def match_names_to_issuer(openfigi_df, csv_df):
    matched = []
    csv_names = csv_df['NAMEOFISSUER'].dropna().tolist()

    for _, row in openfigi_df.iterrows():
        figi_name = row['name']
        if not figi_name:
            best_match = None
        else:
            best_match, score = process.extractOne(figi_name, csv_names)
            if score < 80:  # You can adjust the threshold
                best_match = None
        matched.append(best_match)
        print(f"Processed {figi_name}. {_}")

    openfigi_df['matched_issuer'] = matched
    return openfigi_df

figi_df = pd.DataFrame({'FIGI': ['BBG000DWG505', 'BBG001S5P4H1'], 'name': ['BERKSHIRE HATHAWAY INC', 'BROWN FORMAN CORP'],
                        'ticker': ['BRK-B', 'BF-B']})
csv_df = pd.read_csv('sp500_random.csv')

result_df = match_names_to_issuer(figi_df, csv_df)

# Check which tickers were not matched
unmatched = result_df[result_df['matched_issuer'].isna()]
print(f"Unmatched tickers: {len(unmatched)}")
print(unmatched[['ticker', 'name']])

C:\Users\emadi\AppData\Local\Temp\ipykernel_12128\3996612874.py:24: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_df = pd.read_csv('sp500_random.csv')


Processed BERKSHIRE HATHAWAY INC. 0
Processed BROWN FORMAN CORP. 1
Unmatched tickers: 0
Empty DataFrame
Columns: [ticker, name]
Index: []
